In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1. Packaging Training and Test Data with DataLoader (Batching)
# We use DataLoader so the GPU can process the data in batches
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False) # shuffle=False is required for time series!

# 2. Loss Function and Optimization
criterion = nn.MSELoss()  # MSE (Mean Squared Error) for regression error
optimizer_lstm = optim.Adam(lstm_model.parameters(), lr=0.001)

# 3. Training the LSTM Model (Training Loop)
print("LSTM Model Training Started...")
epochs = 5  # We assign 5 epochs for the initial test (can be increased)
lstm_model.train()

for epoch in range(epochs):
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Forward pass
        outputs = lstm_model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backpropagation and Weight Update
        optimizer_lstm.zero_grad()
        loss.backward()
        optimizer_lstm.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} - Average Training Loss: {total_loss/len(train_loader):.4f}")

# 4. Making Initial Predictions on Test Data
print("\nMaking Predictions on Test Data...")
lstm_model.eval()
with torch.no_grad():
    X_test_device = X_test_tensor.to(device)
    predictions = lstm_model(X_test_device)
    
print("Prediction Process Completed!")
print(f"Predicted Output Shape: {predictions.shape}")
print("First 5 Prediction Values:\n", predictions[:5].cpu().numpy())

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

# 1. Data Loading and Preprocessing
df = pd.read_csv('time_series_60min_singleindex.csv', parse_dates=['utc_timestamp'], index_col='utc_timestamp')
solar_data = df[['AT_solar_generation_actual']].dropna()

scaler = MinMaxScaler(feature_range=(-1, 1))
solar_data_scaled = scaler.fit_transform(solar_data.values)

# 2. Sliding Window
def create_sequences(data, lookback):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:(i + lookback)])
        y.append(data[i + lookback])
    return np.array(X), np.array(y)

lookback = 24 
X, y = create_sequences(solar_data_scaled, lookback)

train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# 3. Tensor Conversions
X_train_tensor = torch.from_numpy(X_train).float()
y_train_tensor = torch.from_numpy(y_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_test_tensor = torch.from_numpy(y_test).float()

# 4. Model Architectures (LSTM and GRU)
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

device = "cuda" if torch.cuda.is_available() else "cpu"
lstm_model = LSTMModel(input_dim=1, hidden_dim=32, layer_dim=2, output_dim=1).to(device)

# 5. Training Loop
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)

criterion = nn.MSELoss()
optimizer_lstm = optim.Adam(lstm_model.parameters(), lr=0.001)

print("LSTM Model Training Started...")
epochs = 5
lstm_model.train()

for epoch in range(epochs):
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        outputs = lstm_model(batch_X)
        loss = criterion(outputs, batch_y)
        
        optimizer_lstm.zero_grad()
        loss.backward()
        optimizer_lstm.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} - Average Training Loss: {total_loss/len(train_loader):.4f}")

# 6. Making Predictions
print("\nMaking Predictions on Test Data...")
lstm_model.eval()
with torch.no_grad():
    X_test_device = X_test_tensor.to(device)
    predictions = lstm_model(X_test_device)
    
print("Prediction Process Completed!")
print(f"Predicted Output Shape: {predictions.shape}")
print("First 5 Prediction Values:\n", predictions[:5].cpu().numpy())